# CMVO — Corrected Expected Return (μ)

**Fix:** Previous notebooks computed μ as `(forecast - actual) / actual`, which measures forecast error using the future realised price. This is not a forward-looking return signal.

**Corrected formula:** `μ = (forecast - current_price) / current_price`

Where `current_price` is the coin price at the time of prediction (rebalance day), and `forecast` is the model's predicted price 7 days ahead. This gives a true expected return estimate that CMVO can meaningfully optimise over.

In [ ]:
import os
print(os.listdir('Results New'))

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# PATHS
PRICES_DIR    = "klines csv data/prices_cleaned"
XGB_BASE_PATH = "Results New/XGBoost Base Forecast new return.npy"
XGB_FEAT_PATH = "Results New/XGBoost Features Forecast new return.npy"
XGB_BO_PATH   = "Results New/Features and BO new return.npy"
ARIMA_PATH    = "Results New/ARIMA_test_7d_rebalance_forecast_matrix.npy"
LSTM_PATH     = "Results New/lstm_cov_matrices.npy"
DCC_PATH      = "Results New/dcc_garch_daily_covariance_matrices_test.npy"
PERF_DIR      = "15 CMVO results corrected"

os.makedirs(PERF_DIR, exist_ok=True)

In [ ]:
# 1. LOAD PRICE DATA
all_coins = sorted(os.listdir(PRICES_DIR))
all_coins = [c for c in all_coins if not c.endswith('.keras')
             and not c.endswith('.npy') and not c.endswith('.csv')]

glimpse  = pd.read_csv(PRICES_DIR + "/" + all_coins[0])
combined = pd.DataFrame({
    'date': pd.date_range(start='2022-04-01', periods=len(glimpse), freq='D')
})
for coin in all_coins:
    combined[coin] = pd.read_csv(PRICES_DIR + "/" + coin).sort_values(by='time')['close'].values

combined = combined.set_index('date')
print(f"Price data shape: {combined.shape}")
print(f"Coins ({len(all_coins)}): {all_coins}")
print(f"Date range: {combined.index[0].date()} to {combined.index[-1].date()}")

# 80/20 split — consistent with DCC and XGBoost
total_rows  = len(combined)
test_start  = int(total_rows * 0.8)
test_prices = combined.iloc[test_start:].copy()
print(f"\nTrain rows: {test_start}")
print(f"Test rows:  {len(test_prices)}")
print(f"Test period: {test_prices.index[0].date()} to {test_prices.index[-1].date()}")

In [ ]:
# 2. LOAD XGB FORECAST PRICES (not returns_predicted)
# Returns shape (n_rebal, n_coins) of raw forecast prices.
def parse_xgb_forecasts(path, all_coins):
    xgb_raw = np.load(path, allow_pickle=True)
    df = pd.DataFrame(xgb_raw)

    if df.shape[1] == 1:
        df = df[0].astype(str).str.split(",", expand=True)

    df = df.iloc[:, 0:4].copy()
    df.columns = ['coin', 'actual', 'forecast', 'returns_predicted']
    df['coin']     = df['coin'].astype(str).str.strip()
    df = df[df['coin'].str.lower() != 'coin'].reset_index(drop=True)
    df['forecast'] = pd.to_numeric(df['forecast'], errors='coerce')

    forecast_list = []
    for coin in all_coins:
        coin_data = df[df['coin'] == str(coin).strip()]['forecast'].values
        coin_data = coin_data[~np.isnan(coin_data)]
        forecast_list.append(coin_data)

    non_empty = [x for x in forecast_list if len(x) > 0]
    if len(non_empty) == 0:
        return np.array([])

    min_len = min(len(x) for x in non_empty)
    return np.column_stack([x[:min_len] for x in non_empty])


# Compute μ = (forecast - current_price) / current_price
# current_price is the coin price on the rebalance day itself (not the future actual).
def compute_mu(forecast_prices, test_prices, all_coins, rebal_days=7):
    n_rebal = forecast_prices.shape[0]
    mu = np.zeros_like(forecast_prices, dtype=float)

    for k in range(n_rebal):
        current_idx = k * rebal_days
        if current_idx < len(test_prices):
            current_prices = test_prices[all_coins].iloc[current_idx].values
            mu[k] = (forecast_prices[k] - current_prices) / current_prices

    return mu

In [ ]:
# 3. LOAD AND COMPUTE CORRECTED MU FOR EACH XGB MODEL
REBAL_DAYS = 7

print("Loading XGBoost Base forecasts...")
fc_base     = parse_xgb_forecasts(XGB_BASE_PATH, all_coins)
mu_base     = compute_mu(fc_base, test_prices, all_coins, REBAL_DAYS)
print(f"mu_base shape: {mu_base.shape}  min: {mu_base.min():.4f}  max: {mu_base.max():.4f}")

print("\nLoading XGBoost Features forecasts...")
fc_features  = parse_xgb_forecasts(XGB_FEAT_PATH, all_coins)
mu_features  = compute_mu(fc_features, test_prices, all_coins, REBAL_DAYS)
print(f"mu_features shape: {mu_features.shape}  min: {mu_features.min():.4f}  max: {mu_features.max():.4f}")

print("\nLoading XGBoost BO Features forecasts...")
fc_bo       = parse_xgb_forecasts(XGB_BO_PATH, all_coins)
mu_bo       = compute_mu(fc_bo, test_prices, all_coins, REBAL_DAYS)
print(f"mu_bo shape: {mu_bo.shape}  min: {mu_bo.min():.4f}  max: {mu_bo.max():.4f}")

In [ ]:
# 4. LOAD COVARIANCE MATRICES AND ALIGN

# ── LSTM ─────────────────────────────────────────────
print("Loading LSTM covariance matrices...")
lstm_full = np.load(LSTM_PATH, allow_pickle=True)
print(f"Full LSTM shape: {lstm_full.shape}")

lstm_test = lstm_full[test_start:]
lstm_cov  = lstm_test[::REBAL_DAYS]
print(f"LSTM test period shape: {lstm_test.shape}")
print(f"LSTM rebalance matrices: {lstm_cov.shape}")

# ── DCC ──────────────────────────────────────────────
print("\nLoading DCC-GARCH covariance matrices...")
dcc_full = np.load(DCC_PATH, allow_pickle=True)
dcc_cov  = dcc_full[::REBAL_DAYS]
print(f"DCC full shape: {dcc_full.shape}")
print(f"DCC rebalance matrices: {dcc_cov.shape}")

# ── ARIMA ─────────────────────────────────────────────
print("\nLoading ARIMA predictions...")
arima_mu = np.load(ARIMA_PATH)
print(f"arima_mu shape: {arima_mu.shape}  min: {arima_mu.min():.4f}  max: {arima_mu.max():.4f}")

# ── ALIGNMENT ─────────────────────────────────────────
test_rows       = len(test_prices)
expected_rebals = (test_rows - 1) // REBAL_DAYS
print(f"\nTest rows: {test_rows}, Expected rebalances: {expected_rebals}")

num_rebal = min(
    mu_base.shape[0],
    mu_features.shape[0],
    mu_bo.shape[0],
    lstm_cov.shape[0],
    dcc_cov.shape[0],
    expected_rebals,
)
print(f"Aligned rebalances: {num_rebal}")

mu_base     = mu_base[:num_rebal]
mu_features = mu_features[:num_rebal]
mu_bo       = mu_bo[:num_rebal]
lstm_cov    = lstm_cov[:num_rebal]
dcc_cov     = dcc_cov[:num_rebal]
arima_mu    = arima_mu[:num_rebal]

print(f"\nmu_base shape:     {mu_base.shape}")
print(f"mu_features shape: {mu_features.shape}")
print(f"mu_bo shape:       {mu_bo.shape}")
print(f"arima_mu shape:    {arima_mu.shape}")
print(f"lstm_cov shape:    {lstm_cov.shape}")
print(f"dcc_cov shape:     {dcc_cov.shape}")

In [ ]:
# 5. CMVO SOLVER
def solve_cmvo(mu, cov, risk_aversion=1.0):
    n       = len(mu)
    cov_reg = cov + 1e-8 * np.eye(n)

    mu_scale  = np.abs(mu).mean()
    cov_scale = np.diag(cov_reg).mean()
    scale     = mu_scale / cov_scale if cov_scale > 0 else 1.0
    cov_scaled = cov_reg * scale

    def neg_obj(w):
        return -(w @ mu - (risk_aversion / 2.0) * (w @ cov_scaled @ w))

    res = minimize(
        neg_obj,
        x0          = np.full(n, 1.0 / n),
        method      = 'SLSQP',
        bounds      = [(-0.1, 0.3)] * n,
        constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
        options     = {'ftol': 1e-9, 'maxiter': 1000},
    )

    if res.success:
        w  = np.clip(res.x, 0.0, 0.3)
        w /= w.sum()
        return w
    return np.full(n, 1.0 / n)

In [ ]:
# 6. PORTFOLIO SIMULATION
def run_portfolio(test_prices, all_coins, mu_array, cov_array,
                  risk_aversion=1.0, cost=None):
    cost       = cost if cost else 0.0
    prices     = test_prices[all_coins].values
    n          = len(all_coins)
    REBAL_DAYS = 7

    holdings         = (1.0 / n) / prices[0]
    portfolio_values = [1.0]
    pct_change       = [0.0]
    weight_history   = [(holdings * prices[0]) / 1.0]
    rebal_count      = 0

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            if rebal_count < len(mu_array):
                mu  = mu_array[rebal_count]
                cov = cov_array[rebal_count]
                new_weights = solve_cmvo(mu, cov, risk_aversion)
                rebal_count += 1
            else:
                new_weights = np.full(n, 1.0 / n)

            current_coin_values = holdings * prices[i]
            target_coin_values  = new_weights * current_value
            trade_amounts       = np.abs(current_coin_values - target_coin_values)
            total_tc            = (trade_amounts * cost).sum()

            net_value = current_value - total_tc
            holdings  = (new_weights * net_value) / prices[i]
            current_value = net_value

        daily_weights = (holdings * prices[i]) / current_value
        weight_history.append(daily_weights)
        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    df_perf    = pd.DataFrame({'portfolio_values': portfolio_values, 'pct_change': pct_change},
                               index=test_prices.index)
    df_weights = pd.DataFrame(weight_history, columns=all_coins, index=test_prices.index)
    return pd.concat([df_perf, df_weights], axis=1)

In [ ]:
# 7. METRICS
def calculate_metrics(result):
    pct_changes      = result['pct_change'].iloc[1:]
    portfolio_values = result['portfolio_values']

    cumulative   = (1 + pct_changes).cumprod()
    rolling_peak = cumulative.cummax()
    drawdown     = (cumulative - rolling_peak) / rolling_peak
    mdd          = drawdown.min()

    mean   = pct_changes.mean()
    std    = pct_changes.std()
    sharpe = (mean / std) if std != 0 else 0.0

    total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) - 1
    calmar       = total_return / abs(mdd) if mdd != 0 else 0.0

    downside = pct_changes[pct_changes < 0]
    down_std = downside.std()
    sortino  = (mean / down_std) if down_std != 0 else 0.0

    return {
        'final_value':  portfolio_values.iloc[-1],
        'total_return': total_return,
        'mdd':          mdd,
        'sharpe':       sharpe,
        'sortino':      sortino,
        'calmar':       calmar,
    }


def save_metrics_npy(metrics_dict, full_path):
    folder_path = os.path.dirname(full_path)
    if folder_path and not os.path.exists(folder_path):
        os.makedirs(folder_path, exist_ok=True)
    np.save(full_path, metrics_dict)
    print(f"Saved: {full_path}")

In [ ]:
# 8. RUN ALL MODELS (cost = 0.1%)
all_metrics = {}

for model_name, mu_array, cov in [
    ('xgb_base_lstm',     mu_base,     lstm_cov),
    ('xgb_features_lstm', mu_features, lstm_cov),
    ('xgb_bo_lstm',       mu_bo,       lstm_cov),
    ('xgb_base_dcc',      mu_base,     dcc_cov),
    ('xgb_features_dcc',  mu_features, dcc_cov),
    ('xgb_bo_dcc',        mu_bo,       dcc_cov),
    ('arima_lstm',        arima_mu,    lstm_cov),
    ('arima_dcc',         arima_mu,    dcc_cov),
]:
    res = run_portfolio(test_prices, all_coins, mu_array, cov, cost=0.001)
    m   = calculate_metrics(res)
    print(f"=== {model_name} (cost=0.001) ===")
    print(f"  Final value:  {m['final_value']:.4f}")
    print(f"  Total return: {m['total_return']:.4f}")
    print(f"  MDD:          {m['mdd']:.4f}")
    print(f"  Sharpe:       {m['sharpe']:.4f}")
    print(f"  Sortino:      {m['sortino']:.4f}")
    print(f"  Calmar:       {m['calmar']:.4f}")
    print()
    all_metrics[model_name] = m
    res.to_csv(os.path.join(PERF_DIR, f'{model_name}.csv'), index=False)

save_metrics_npy(all_metrics, os.path.join(PERF_DIR, 'all_portfolio_metrics.npy'))

In [ ]:
# 9. BENCHMARKS
def run_equal_weight(test_prices, all_coins):
    n        = len(all_coins)
    prices   = test_prices[all_coins].values
    holdings = (1.0 / n) / prices[0]

    portfolio_values = [1.0]
    pct_change       = [0.0]

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()
        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    return pd.DataFrame({'portfolio_values': portfolio_values, 'pct_change': pct_change})


def run_mv_benchmark(test_prices, all_coins, lookback=60, risk_aversion=1.0, cost=0.001):
    n               = len(all_coins)
    prices          = test_prices[all_coins].values
    dates           = test_prices.index
    current_weights = np.full(n, 1.0 / n)
    holdings        = current_weights / prices[0]
    portfolio_values = [1.0]
    pct_change       = [0.0]
    weight_history   = [current_weights.copy()]
    REBAL_DAYS       = 7

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            start          = max(0, i - lookback)
            window_prices  = prices[start:i]
            window_returns = np.diff(window_prices, axis=0) / window_prices[:-1]

            if len(window_returns) >= 2:
                mu_hist         = window_returns.mean(axis=0)
                cov_hist        = np.cov(window_returns.T) + 1e-8 * np.eye(n)
                current_weights = solve_cmvo(mu_hist, cov_hist, risk_aversion)
            else:
                current_weights = np.full(n, 1.0 / n)

            current_coin_values = holdings * prices[i]
            target_coin_values  = current_weights * current_value
            trade_amounts       = np.abs(current_coin_values - target_coin_values)
            total_tc            = (trade_amounts * cost).sum()
            net_value           = current_value - total_tc
            holdings            = (current_weights * net_value) / prices[i]
            current_value       = net_value
        else:
            current_weights = (holdings * prices[i]) / current_value

        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)
        weight_history.append(current_weights.copy())

    results_df  = pd.DataFrame({'date': dates, 'portfolio_values': portfolio_values, 'pct_change': pct_change})
    weights_df  = pd.DataFrame(weight_history, columns=all_coins)
    return pd.concat([results_df, weights_df], axis=1).set_index('date')


res_1n = run_equal_weight(test_prices, all_coins)
m_1n   = calculate_metrics(res_1n)

res_mv = run_mv_benchmark(test_prices, all_coins, cost=0.001)
m_mv   = calculate_metrics(res_mv)

all_metrics['1/N Equal Weight'] = m_1n
all_metrics['Statistical MV']   = m_mv
save_metrics_npy(all_metrics, os.path.join(PERF_DIR, 'all_portfolio_metrics.npy'))

res_1n.to_csv(os.path.join(PERF_DIR, 'benchmark_1n.csv'), index=False)
res_mv.to_csv(os.path.join(PERF_DIR, 'benchmark_mv.csv'), index=False)

In [ ]:
# 10. FULL COMPARISON TABLE
print("=== FULL RESULTS (test period: Apr–Dec 2025, cost=0.001, corrected μ) ===")
print(f"{'Model':<25} {'Final':>8} {'Return':>8} {'Sharpe':>8} {'MDD':>8} {'Sortino':>8} {'Calmar':>8}")
print("-" * 80)

for name, m in [('1/N Equal Weight', m_1n), ('Statistical MV', m_mv)]:
    print(f"{name:<25} {m['final_value']:>8.4f} {m['total_return']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

print("-" * 80)

for model_name, m in all_metrics.items():
    if model_name not in ('1/N Equal Weight', 'Statistical MV'):
        print(f"{model_name:<25} {m['final_value']:>8.4f} {m['total_return']:>8.4f} {m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

# TESTING FOR SHORTING

In [ ]:
# 11. BOUND SEARCH (xgb_bo_lstm, cost=0.001)
long_bounds  = [0.20, 0.25, 0.30, 0.40, 0.50]
short_bounds = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]

print("=== BOUND SEARCH (xgb_bo_lstm, cost=0.001) ===")
print(f"{'Long':>6} {'Short':>6} {'Final':>8} {'Sharpe':>8} {'MDD':>8} {'Calmar':>8}")
print("-" * 50)

best_sharpe = -np.inf
best_long   = None
best_short  = None

for long_b in long_bounds:
    for short_b in short_bounds:

        if len(all_coins) * long_b < 1.0:
            print(f"{long_b:>6.2f} {short_b:>6.2f} {'INFEASIBLE':>8}")
            continue

        def solve_test(mu, cov, lb=long_b, sb=short_b):
            n          = len(mu)
            cov_reg    = cov + 1e-8 * np.eye(n)
            mu_scale   = np.abs(mu).mean()
            cov_scale  = np.diag(cov_reg).mean()
            scale      = mu_scale / cov_scale if cov_scale > 0 else 1.0
            cov_scaled = cov_reg * scale
            def neg_obj(w):
                return -(w @ mu - (1.0 / 2.0) * (w @ cov_scaled @ w))
            res = minimize(
                neg_obj,
                x0          = np.full(n, 1.0 / n),
                method      = 'SLSQP',
                bounds      = [(-sb, lb)] * n,
                constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
                options     = {'ftol': 1e-9, 'maxiter': 1000},
            )
            if res.success:
                w = np.clip(res.x, -sb, lb)
                w /= w.sum()
                return w
            return np.full(n, 1.0 / n)

        prices      = test_prices[all_coins].values
        n           = len(all_coins)
        holdings    = (1.0 / n) / prices[0]
        pv, pc      = [1.0], [0.0]
        rebal_count = 0

        for i in range(1, len(prices)):
            current_value = (holdings * prices[i]).sum()
            if i % 7 == 0:
                if rebal_count < len(mu_bo):
                    new_weights = solve_test(mu_bo[rebal_count], lstm_cov[rebal_count])
                    rebal_count += 1
                else:
                    new_weights = np.full(n, 1.0 / n)
                tc            = (np.abs(holdings * prices[i] - new_weights * current_value) * 0.001).sum()
                net_value     = current_value - tc
                holdings      = (new_weights * net_value) / prices[i]
                current_value = net_value
            pv.append(current_value)
            pc.append((current_value / pv[-2]) - 1.0)

        res_df = pd.DataFrame({'portfolio_values': pv, 'pct_change': pc})
        m      = calculate_metrics(res_df)

        flag = ' ← best' if m['sharpe'] > best_sharpe and m['mdd'] > -1.0 else ''
        if m['sharpe'] > best_sharpe and m['mdd'] > -1.0:
            best_sharpe = m['sharpe']
            best_long   = long_b
            best_short  = short_b

        mdd_flag = ' ⚠' if m['mdd'] < -1.0 else ''
        print(f"{long_b:>6.2f} {short_b:>6.2f} {m['final_value']:>8.4f} "
              f"{m['sharpe']:>8.4f} {m['mdd']:>8.4f}{mdd_flag} {m['calmar']:>8.4f}{flag}")

print()
print(f"Best: long={best_long}, short={best_short}, Sharpe={best_sharpe:.4f}")
print(f"Note: ⚠ means MDD < -1.0 (portfolio went negative — unrealistic, excluded)")

In [ ]:
# 12. SAVE BEST MODEL PREDICTIONS (xgb_bo + lstm)
mu_df = pd.DataFrame(mu_bo, columns=all_coins)
print(mu_df.head())

n_periods, n_assets, _ = lstm_cov.shape
print(f"Min Covariance in tensor: {lstm_cov.min()}")
print(f"Max Covariance in tensor: {lstm_cov.max()}")

iu = np.triu_indices(n_assets, k=1)
il = np.tril_indices(n_assets, k=-1)

avg_corr_per_period = np.array([
    matrix[np.concatenate((iu[0], il[0])), np.concatenate((iu[1], il[1]))].mean()
    for matrix in lstm_cov
])
avg_cov = pd.DataFrame(avg_corr_per_period)
print(f"Average Covariance for first 3 periods: {avg_corr_per_period[:3]}")

os.makedirs("15 CMVO results corrected/bestmodelpredictions", exist_ok=True)
mu_df.to_csv("15 CMVO results corrected/bestmodelpredictions/xgb_bo_returns_corrected.csv", index=False)
avg_cov.to_csv("15 CMVO results corrected/bestmodelpredictions/lstm_correlations.csv", index=False)
print("Saved corrected mu and LSTM correlations.")